In [ ]:
import os, sys, warnings
from pathlib import Path
_r = Path.cwd()
while not (_r / 'requirements.txt').exists() and _r != _r.parent:
    _r = _r.parent
os.chdir(_r); sys.path.insert(0, str(_r / 'src')); warnings.filterwarnings('ignore')

# Fleet warranty-risk table — projected SoH at warranty expiry

For every vehicle we free-run the **XGBoost condition-aware degradation model** forward from its last
observed month to its **warranty boundary** (per-model years *and* km, whichever comes first — from
`src/config.warranty_for`), then read off the projected SoH. A vehicle is **at risk** if its SoH is
projected to cross **80%** before warranty expiry.

This directly tests the working assumption that *~90% of the fleet survives the warranty period
without degrading to 80%*.

In [ ]:
import numpy as np, pandas as pd, xgboost as xgb
import matplotlib.pyplot as plt
import model, config

m = pd.read_parquet('data/mahindra/features/feature_table.parquet').sort_values(['vin','month'])
FEATS, STATE, STRESS = model.FEATS, model.STATE, model.STRESS
tr_all = model.build_transitions(m)
xgb_full = xgb.XGBRegressor(n_estimators=300, learning_rate=0.03, max_depth=4, subsample=0.8,
                            colsample_bytree=0.8, n_jobs=8, verbosity=0).fit(
    tr_all[FEATS].to_numpy(), tr_all['loss'].to_numpy(), sample_weight=tr_all['w'].to_numpy())

# vin -> model (cached) and registration date
vmod = dict(pd.read_csv('data/manifests/mahindra_vin_model.csv').values)
reg_df = pd.read_csv('Mh_Regd_Date.csv'); reg_df['reg'] = pd.to_datetime(reg_df['vehicle_registration_date'], errors='coerce')
REG = dict(zip(reg_df['vin'], reg_df['reg']))
print(f'{m.vin.nunique()} vehicles | model trained on {len(tr_all)} transitions')

In [ ]:
def free_run(g, mdl, max_age):
    """Roll SoH + odometer forward from the last observed month (recent-median stress persists)."""
    g = g.sort_values('month'); last = g.iloc[-1]
    stress = g.iloc[-6:][STRESS].median().to_dict()
    st = {s: float(last[s]) for s in STATE}
    odo0 = float(last['odo_max']);  odo0 = odo0 if np.isfinite(odo0) else float(np.nan_to_num(g['odo_max'].max()))
    st['odo_max'] = odo0
    soh = float(last['soh']); odo = odo0; kmpm = max(float(np.nan_to_num(g['km_month'].iloc[-6:].median())), 0.0)
    ages=[st['age_months']]; sohs=[soh]; odos=[odo]
    while ages[-1] < max_age and soh > 50:
        isa, dfc = model._curv(st['age_months'], st['soh'])
        x = pd.DataFrame([{**{s: st[s] for s in STATE}, **stress, 'inv_sqrt_age': isa, 'soh_deficit': dfc}])[FEATS].to_numpy()
        soh = soh - max(mdl.predict(x)[0], 0); odo = odo + kmpm
        st.update(soh=soh, age_months=st['age_months']+1, cum_ah=st['cum_ah']+stress.get('ah_throughput',0),
                  cum_km=st['cum_km']+kmpm, odo_max=odo)
        ages.append(st['age_months']); sohs.append(soh); odos.append(odo)
    return np.array(ages), np.array(sohs), np.array(odos)

I = lambda v: int(round(v)) if (v is not None and np.isfinite(v)) else None   # nan-safe int
MIN_HIST = 8                                                                   # months of history for a reliable forecast
rows=[]
for vin, g in m.groupby('vin'):
    g = g.sort_values('age_months'); mdl_name = vmod.get(vin, '?')
    wyr, wkm = config.warranty_for('mahindra', mdl_name)
    warr_age = wyr*12.0
    age_now = float(g['age_months'].iloc[-1]); soh_now = float(g['soh'].iloc[-1]); n_obs = len(g)
    odo_now = float(np.nan_to_num(g['odo_max'].max()))     # cumulative odometer (nan-safe)
    # full trajectory = observed + forecast continuation to (a bit past) the time-warranty boundary
    fa, fs, fo = free_run(g, xgb_full, max(warr_age, age_now)+1)
    A = np.concatenate([g['age_months'].to_numpy(), fa[1:]]); S = np.concatenate([g['soh'].to_numpy(), fs[1:]])
    Oraw = np.concatenate([g['odo_max'].to_numpy(), fo[1:]])
    order = np.argsort(A); A,S,Oraw = A[order], S[order], Oraw[order]
    O = pd.Series(Oraw).ffill().bfill().to_numpy()         # fill odometer gaps for the km check
    # km-limited warranty age (first age odo crosses the km cap), else inf
    km_age = A[np.argmax(O>=wkm)] if (O>=wkm).any() else np.inf
    eff_age = min(warr_age, km_age)                        # warranty ends at the earlier of time / km
    proj_soh = float(np.interp(eff_age, A, S))             # projected SoH at warranty expiry
    proj_odo = float(np.interp(warr_age, A, O))
    breach_age = A[np.argmax(S<80)] if (S<80).any() else np.inf   # first age SoH<80
    horizon = warr_age - age_now                                  # months we must forecast forward
    conf = 'low' if (n_obs < MIN_HIST or mdl_name == 'Ape E-Xtra FX' or horizon > 2*max(age_now,1)) else 'ok'
    reg = REG.get(vin)
    rows.append(dict(vin=vin, model=mdl_name, warr_yr=wyr, warr_km=wkm, n_months=n_obs, conf=conf,
                     age_now_mo=round(age_now,1), soh_now=round(soh_now,1), odo_km=I(odo_now),
                     proj_soh_at_warr=round(proj_soh,1),
                     limit=('km' if km_age<warr_age else 'time'),
                     proj_km_at_warr=I(proj_odo),
                     breach=('YES' if proj_soh<80 else 'no'),
                     age_at_80_yr=(round(breach_age/12,1) if np.isfinite(breach_age) else None),
                     date_80=((reg + pd.Timedelta(days=breach_age*30.4375)).date() if (reg is not None and pd.notna(reg) and np.isfinite(breach_age)) else None),
                     warr_end=((reg + pd.DateOffset(years=int(wyr))).date() if (reg is not None and pd.notna(reg)) else None)))
risk = pd.DataFrame(rows).sort_values('proj_soh_at_warr').reset_index(drop=True)
risk.to_csv('data/mahindra/soh/warranty_risk.csv', index=False)
print(f'built warranty-risk table for {len(risk)} vehicles -> data/mahindra/soh/warranty_risk.csv')

## Fleet summary — does the 90%-survival assumption hold?

In [ ]:
surv = (risk['proj_soh_at_warr']>=80).mean()*100
rel = risk[risk['conf']=='ok']
surv_rel = (rel['proj_soh_at_warr']>=80).mean()*100
print(f"Projected to survive warranty (>=80% at expiry): {surv:.0f}% of all 95   [assumption: 90%]")
print(f"   ... of {len(rel)} reliable vehicles (>=8 mo history): {surv_rel:.0f}%")
print(f"At-risk (projected <80% before warranty end):    {(risk['breach']=='YES').sum()} of 95  "
      f"({(rel['breach']=='YES').sum()} reliable, {(risk[risk['conf']=='low']['breach']=='YES').sum()} low-confidence)")
print(f"Median projected SoH at warranty expiry:          {risk['proj_soh_at_warr'].median():.0f}%\n")
by = risk.groupby('model').agg(n=('vin','size'), warr_yr=('warr_yr','first'),
        survive_pct=('proj_soh_at_warr', lambda s:(s>=80).mean()*100),
        med_proj_soh=('proj_soh_at_warr','median'), at_risk=('breach', lambda s:(s=='YES').sum()))
print(by.round(0).to_string())
print(f"\nLimiting factor: km-first {int((risk['limit']=='km').sum())}, time-first {int((risk['limit']=='time').sum())}"
      f"  (low utilization -> calendar+condition aging, not mileage, drives warranty risk)")

## At-risk vehicles — worst first (projected to breach 80% within warranty)

In [ ]:
cols=['vin','model','warr_yr','n_months','conf','age_now_mo','soh_now','odo_km','proj_soh_at_warr','age_at_80_yr','date_80','warr_end']
atrisk = risk[risk['breach']=='YES'][cols]
print(f"{len(atrisk)} at-risk vehicles  ({(atrisk['conf']=='ok').sum()} reliable, {(atrisk['conf']=='low').sum()} low-confidence / short-history):\n")
print(atrisk.to_string(index=False))
print("\nconf='low' = <8 mo history, an Ape (mislabeled Piaggio), or warranty horizon > 2x current age "
      "-> projection is an extrapolation, treat as a watch-list not a firm call.")

## Visualization — projected SoH at warranty expiry

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5.2))
# (a) every vehicle's projected SoH at warranty, sorted, colored by risk
r = risk.sort_values('proj_soh_at_warr').reset_index(drop=True)
colors = np.where(r['proj_soh_at_warr']<80, 'tab:red', 'tab:green')
axes[0].bar(range(len(r)), r['proj_soh_at_warr'], color=colors, width=1.0)
axes[0].axhline(80, ls='--', color='k', lw=1); axes[0].text(2, 81, '80% EOL', fontsize=9)
axes[0].set_xlabel('vehicles (sorted worst -> best)'); axes[0].set_ylabel('projected SoH at warranty expiry (%)')
axes[0].set_title(f"Projected SoH at warranty expiry  |  {(r['proj_soh_at_warr']>=80).mean()*100:.0f}% survive (assumption 90%)")
axes[0].set_ylim(min(60, r['proj_soh_at_warr'].min()-3), 101)
# (b) survival rate by model vs 90% target
by = risk.groupby('model').agg(survive=('proj_soh_at_warr', lambda s:(s>=80).mean()*100), n=('vin','size'))
axes[1].bar(by.index, by['survive'], color=['tab:green' if v>=90 else 'tab:orange' for v in by['survive']])
axes[1].axhline(90, ls='--', color='k', lw=1); axes[1].text(0, 91, '90% target', fontsize=9)
for i,(v,n) in enumerate(zip(by['survive'], by['n'])): axes[1].text(i, v+1, f'{v:.0f}%\n(n={n})', ha='center', fontsize=9)
axes[1].set_ylabel('% surviving warranty (>=80%)'); axes[1].set_ylim(0,105); axes[1].set_title('Warranty survival by model')
plt.tight_layout(); plt.show()

In [ ]:
from matplotlib.lines import Line2D
# All 95 vehicles on one axis vs age since registration: actual (grey) + forecast colored by
# warranty outcome. GREEN = stays >=80% through warranty. At-risk (crosses 80% before warranty)
# is split by model: PURPLE = Treo, GOLD = other variant (Zor Grand / Ape).
breach_map = dict(zip(risk['vin'], risk['breach']))
TARGET_AGE = 60.0  # forecast every vehicle to 5 years of age
fig, ax = plt.subplots(figsize=(13, 7))
n_green = n_treo = n_other = 0
for vin, g in m.groupby('vin'):
    g = g.sort_values('age_months')
    aa = g['age_months'].to_numpy() / 12.0; ss = g['soh'].to_numpy()
    at_risk = breach_map.get(vin) == 'YES'
    is_treo = vmod.get(vin) == 'Treo'
    if not at_risk:
        fcolor = 'tab:green'; z = 2; n_green += 1
    elif is_treo:
        fcolor = 'tab:purple'; z = 3; n_treo += 1
    else:
        fcolor = 'gold'; z = 3; n_other += 1
    ax.plot(aa, ss, '-', color='0.6', lw=0.8, alpha=0.35, zorder=1)               # actual (grey)
    fa, fs, _ = free_run(g, xgb_full, max(TARGET_AGE, aa[-1]*12 + 1))
    ax.plot(fa / 12.0, fs, '-', color=fcolor, lw=1.1, alpha=0.6, zorder=z)         # forecast
ax.set_xlim(0, 5.2); ax.set_ylim(55, 101); ax.grid(alpha=0.3)
ax.axhline(80, ls='--', color='black', lw=1.2, zorder=4); ax.text(0.05, 80.5, '80% EOL', color='black', fontsize=9)
for yr, lbl in [(3, '3-yr (Treo)'), (5, '5-yr (Zor Grand)')]:
    ax.axvline(yr, ls=':', color='0.3', lw=1.1); ax.text(yr - 0.02, 56, lbl, fontsize=8, color='0.3', ha='right', rotation=90)
ax.set_xlabel('age since registration (years)'); ax.set_ylabel('SoH (%)')
ax.set_title('All 95 vehicles — actual SoH (grey) + XGBoost forecast (green = survives; at-risk: purple Treo / gold other)', fontsize=11.5)
ax.legend(handles=[Line2D([0], [0], color='0.6', lw=2, label='actual (observed)'),
                   Line2D([0], [0], color='tab:green', lw=2, label=f'survives warranty (n={n_green})'),
                   Line2D([0], [0], color='tab:purple', lw=2, label=f'at-risk — Treo (n={n_treo})'),
                   Line2D([0], [0], color='gold', lw=2, label=f'at-risk — other variant (n={n_other})')],
          loc='lower left', fontsize=9)
plt.tight_layout(); plt.show()